# Stage 2 实验 — Step 1：数据探索与 BIO 转换验证

**目标**：
1. 确认 CheckThat! 2024 字符级标注数据路径
2. 理解数据格式（SemEval train → BIO 训练数据）
3. 验证 BIO 转换函数的正确性（抽样20条）
4. 统计训练/评估数据量

**关键说明**：
- 训练数据来源：CheckThat! 2024 的字符级标注（EN/PL/RU 训练集）
- 评估数据来源：SemEval 2023 dev 集（与 CheckThat 训练集有重叠的那部分）
- Gold 标准：使用 CheckThat 2024 的字符级标注作为 gold，而非 SemEval 段落级标注

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory containing data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 / CheckThat!-2024 overlap evaluation set
# Expected subdirectories: en_overlap_articles/, po_overlap_articles/, ru_overlap_articles/
OVERLAP_DIR = "your/path/here"  # typically BASE + "/data_analy/overlap_analysis_results"

# CheckThat! 2024 Task 3 training data bundle
CHECKTHAT24_DIR = CHECKTHAT24_DIR  # Set in configuration cell above  # root of the official CheckThat!-2024 data bundle

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
import os
import glob

BASE = "your/path/here"
NOTEBOOKS_DIR = os.path.join(BASE, 'Total_work', 'research_projects', 'disinformation_detection', 'notebooks')
STAGE1_DIR = os.path.join(NOTEBOOKS_DIR, 'compare_result_F1', 'checkthat2024', 'debug_results')
# [Note] 金标准和文章实际存放于 overlap_analysis_results，
#         TASK2023路径不正确。改动：改为 OVERLAP_DIR 指向实际目录。
OVERLAP_DIR = os.path.join(NOTEBOOKS_DIR, 'data_analy', 'overlap_analysis_results')

# ============================================================
# 1. 先把所有可能相关的目录列出来
# ============================================================
print('=== 探索 TRLAL 目录结构 ===')
for root, dirs, files in os.walk(BASE):
    # 跳过太深的层级
    depth = root.replace(BASE, '').count(os.sep)
    if depth > 3:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 3:
        sub_indent = '  ' * (depth + 1)
        for f in files[:5]:  # 每目录最多显示5个文件
            print(f'{sub_indent}{f}')
        if len(files) > 5:
            print(f'{sub_indent}... ({len(files)} files total)')

In [ ]:
# ============================================================
# 2. 寻找 CheckThat! 2024 数据目录
# ============================================================
print('=== 搜索 checkthat/2024 相关目录 ===')

# 可能的路径模式
search_roots = [
    BASE,
    BASE,
    os.path.join(BASE, 'data'),
]

for root_path in search_roots:
    if not os.path.exists(root_path):
        continue
    for root, dirs, files in os.walk(root_path):
        depth = root.replace(root_path, '').count(os.sep)
        if depth > 4:
            continue
        dir_lower = root.lower()
        if any(kw in dir_lower for kw in ['checkthat', 'check_that', '2024']):
            print(f'  Found: {root}')
            for f in files[:10]:
                print(f'    {f}')
            if len(files) > 10:
                print(f'    ... ({len(files)} files)')

In [ ]:
# ============================================================
# 3. 搜索字符级标注文件（spans/subtask3/task3）
# ============================================================
print('=== 搜索字符级标注文件 ===')

for root, dirs, files in os.walk(BASE):
    depth = root.replace(BASE, '').count(os.sep)
    if depth > 5:
        continue
    for f in files:
        fl = f.lower()
        if any(kw in fl for kw in ['span', 'subtask3', 'task3', 'labels-subtask-3']):
            print(f'  {os.path.join(root, f)}')

In [ ]:
# ============================================================
# 4. 检查已知的 Stage 1 结果文件格式（了解接口）
# ============================================================
import pandas as pd

stage1_files = {
    'en_model':   f'{STAGE1_DIR}/en_model_all_ai_results.tsv',
    'en_context': f'{STAGE1_DIR}/en_context_all_ai_results.tsv',
    'en_voting':  f'{STAGE1_DIR}/en_voting_aggressiv_all_ai_results.tsv',
    'po_model':   f'{STAGE1_DIR}/po_model_all_ai_results.tsv',
    'po_context': f'{STAGE1_DIR}/po_context_all_ai_results.tsv',
    'po_voting':  f'{STAGE1_DIR}/po_voting_aggressiv_all_ai_results.tsv',
    'ru_model':   f'{STAGE1_DIR}/ru_model_all_ai_results.tsv',
    'ru_context': f'{STAGE1_DIR}/ru_context_all_ai_results.tsv',
    'ru_voting':  f'{STAGE1_DIR}/ru_voting_aggressive_all_ai_results.tsv',
}

for name, path in stage1_files.items():
    exists = os.path.exists(path)
    print(f'  {name}: {"✅" if exists else "❌"} {path}')

In [ ]:
# ============================================================
# 5. 查看 Stage 1 结果文件的列结构（选一个存在的）
# ============================================================
for name, path in stage1_files.items():
    if os.path.exists(path):
        df = pd.read_csv(path, sep='\t', nrows=3)
        print(f'\n=== {name} 列结构 ===')
        print(f'Columns: {list(df.columns)}')
        print(df.head(3).to_string())
        break

In [ ]:
# ============================================================
# 6. 检查 SemEval dev 金标准格式
# ============================================================
gold_files = {
    'EN': os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt'),
    'PL': os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'labels-subtask-3-spans.txt'),
    'RU': os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'labels-subtask-3-spans.txt'),
}

for lang, path in gold_files.items():
    exists = os.path.exists(path)
    print(f'  Gold {lang}: {"✅" if exists else "❌"} {path}')
    if exists:
        with open(path, 'r', encoding='utf-8') as f:
            lines = f.readlines()[:5]
        print(f'    首5行: {lines}')

In [ ]:
# ============================================================
# 7. 搜索 CheckThat 2024 的字符级标注（train 集）
#    这是 Stage 2 token classifier 的 **训练数据**
# ============================================================
print('=== 搜索 CheckThat 2024 训练集字符级标注 ===')

# 已知 SemEval dev 有 overlap_analysis 目录
overlap_paths = [
    f'{BASE}/overlap_analysis_results',
    os.path.join(BASE, 'overlap_analysis_results'),
]
for p in overlap_paths:
    if os.path.exists(p):
        print(f'Found overlap dir: {p}')
        for root, dirs, files in os.walk(p):
            for f in files:
                print(f'  {os.path.join(root, f)}')

# 搜索所有 .txt 文件中包含 span 格式的
print('\n=== 搜索形如 article*spans*.txt 的文件 ===')
for root, dirs, files in os.walk(BASE):
    depth = root.replace(BASE, '').count(os.sep)
    if depth > 6:
        continue
    for f in files:
        if 'span' in f.lower() and f.endswith('.txt'):
            print(f'  {os.path.join(root, f)}')

In [ ]:
# ============================================================
# 8. 查看 SemEval 文章文件格式（训练用原始文章）
# ============================================================
article_dirs = {
    'EN': os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'articles'),
    'PL': os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'articles'),
    'RU': os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'articles'),
}
for lang, path in article_dirs.items():
    exists = os.path.exists(path)
    if exists:
        files = os.listdir(path)
        print(f'{lang}: {len(files)} articles — 样例: {files[:3]}')
        # 读取第一篇文章的前200字符
        first = os.path.join(path, files[0])
        with open(first, 'rb') as f:
            content = f.read(300).decode('utf-8', errors='replace')
        print(f'  内容预览（前300字节）:\n  {repr(content)}')
    else:
        print(f'{lang}: ❌ {path}')

In [ ]:
# ============================================================
# 9. 解析金标准格式，理解字符级 span 结构
# ============================================================
# [Note] overlap_analysis_results 中的 labels-subtask-3-spans.txt
#         为字符级标注（4列：article_id 	 technique 	 char_start 	 char_end），
#         之前误解为段落级3列格式。改动：恢复为解析4列字符级span格式。
def parse_gold_spans(gold_path):
    """
    解析 CheckThat! 2024 字符级 span 标注文件
    格式：article_id	technique	char_start	char_end
    返回: {article_id: [(start, end, technique), ...]}
    """
    spans_by_article = {}
    with open(gold_path, 'rb') as f:
        for line in f:
            line_str = line.decode('utf-8', errors='replace').rstrip()
            if not line_str.strip():
                continue
            parts = line_str.split('	')
            if len(parts) != 4:
                print(f'  Unexpected format ({len(parts)} cols): {parts}')
                continue
            article_id, technique, start, end = parts
            try:
                start, end = int(start), int(end)
            except ValueError:
                continue
            if article_id not in spans_by_article:
                spans_by_article[article_id] = []
            spans_by_article[article_id].append((start, end, technique))
    return spans_by_article

en_gold = os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt')
if os.path.exists(en_gold):
    print('=== EN Gold 格式探查 ===')
    with open(en_gold, 'rb') as f:
        for i, line in enumerate(f):
            line_str = line.decode('utf-8', errors='replace').rstrip('\r\n')
            parts = line_str.split('\t')
            print(f'Line {i}: {parts}')
            if i >= 9:
                break

In [ ]:
# ============================================================
# 10. 实现 parse_gold_spans（根据上面观察到的格式填充）
# ============================================================
# [Note] overlap_analysis_results 中的 labels-subtask-3-spans.txt
#         为字符级标注（4列：article_id 	 technique 	 char_start 	 char_end），
#         之前误解为段落级3列格式。改动：恢复为解析4列字符级span格式。
def parse_gold_spans(gold_path):
    """
    解析 CheckThat! 2024 字符级 span 标注文件
    格式：article_id	technique	char_start	char_end
    返回: {article_id: [(start, end, technique), ...]}
    """
    spans_by_article = {}
    with open(gold_path, 'rb') as f:
        for line in f:
            line_str = line.decode('utf-8', errors='replace').rstrip()
            if not line_str.strip():
                continue
            parts = line_str.split('	')
            if len(parts) != 4:
                print(f'  Unexpected format ({len(parts)} cols): {parts}')
                continue
            article_id, technique, start, end = parts
            try:
                start, end = int(start), int(end)
            except ValueError:
                continue
            if article_id not in spans_by_article:
                spans_by_article[article_id] = []
            spans_by_article[article_id].append((start, end, technique))
    return spans_by_article

# 测试
en_gold = os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt')
if os.path.exists(en_gold):
    gold_en = parse_gold_spans(en_gold)
    print(f'EN Gold articles: {len(gold_en)}')
    first_id = list(gold_en.keys())[0]
    print(f'  First article ({first_id}): {gold_en[first_id][:3]}')

In [ ]:
# ============================================================
# 11. BIO 转换函数（核心，需要用 Fast Tokenizer）
# ============================================================
TECHNIQUES = [
    'Appeal_to_Authority', 'Appeal_to_Fear-Prejudice', 'Appeal_to_Hypocrisy',
    'Appeal_to_Popularity', 'Appeal_to_Time', 'Appeal_to_Values',
    'Causal_Oversimplification', 'Consequential_Oversimplification',
    'Conversation_Killer', 'Doubt', 'Exaggeration-Minimisation',
    'False_Dilemma-No_Choice', 'Flag_Waving', 'Guilt_by_Association',
    'Loaded_Language', 'Name_Calling-Labeling', 'Obfuscation-Vagueness-Confusion',
    'Questioning_the_Reputation', 'Red_Herring', 'Repetition',
    'Slogans', 'Straw_Man', 'Whataboutism'
]
TECH2IDX = {t: i for i, t in enumerate(TECHNIQUES)}
NUM_LABELS = 1 + 23 * 2  # O + 23*(B+I) = 47

def convert_to_bio(text, char_spans, tokenizer, max_length=256):
    """
    将字符级 span 标注转换为 BIO token 标签序列
    
    Args:
        text:       文章原始文本（rb模式读入，保持字符偏移一致）
        char_spans: [(start, end, technique), ...]
        tokenizer:  BertTokenizerFast 或 XLMRobertaTokenizerFast
        max_length: 最大token数，超出截断
    
    Returns:
        encoding: tokenizer输出（含offset_mapping）
        labels:   List[int]，长度=max_length，BIO标签
        
    标签编码：
        0           = O
        1 + t*2     = B-<technique_t>    (t=0..22)
        2 + t*2     = I-<technique_t>
    """
    encoding = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_offsets_mapping=True
    )
    offset_mapping = encoding['offset_mapping']
    labels = [0] * max_length  # 全O

    for span_start, span_end, technique in char_spans:
        if technique not in TECH2IDX:
            continue  # 忽略未知技术
        tech_idx = TECH2IDX[technique]
        b_label = 1 + tech_idx * 2   # B-tag
        i_label = 2 + tech_idx * 2   # I-tag

        is_first = True
        for token_idx, (tok_start, tok_end) in enumerate(offset_mapping):
            if tok_start == tok_end:  # 特殊 token（[CLS]/[SEP]/padding）
                continue
            # token 与 span 有重叠
            if tok_start < span_end and tok_end > span_start:
                labels[token_idx] = b_label if is_first else i_label
                is_first = False

    return encoding, labels

print(f'NUM_LABELS = {NUM_LABELS}  (0=O, 1..46 = B/I for 23 techniques)')
print(f'标签示例：')
for i, tech in enumerate(TECHNIQUES[:3]):
    print(f'  {tech}: B={1+i*2}, I={2+i*2}')

In [ ]:
# ============================================================
# 12. 验证 BIO 转换（用 mBERT Fast Tokenizer）
# ============================================================
try:
    from transformers import BertTokenizerFast, XLMRobertaTokenizerFast
    print('transformers 已安装 ✅')
except ImportError:
    print('安装 transformers...')
    import subprocess
    subprocess.run(['pip', 'install', 'transformers', '-q'])
    from transformers import BertTokenizerFast, XLMRobertaTokenizerFast

# 用 mBERT Fast Tokenizer 测试
print('Loading mBERT Fast Tokenizer...')
tokenizer_mbert = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
print('Loading XLM-RoBERTa Fast Tokenizer...')
tokenizer_xlmr  = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')
print('Both tokenisers loaded ✅')

In [ ]:
# ============================================================
# 13. BIO 转换正确性验证（用一篇真实文章）
# ============================================================
import numpy as np

def verify_bio_conversion(article_id, article_dir, gold_spans_dict, tokenizer, max_length=256, verbose=True):
    """
    对单篇文章验证 BIO 转换：
    1. 读取原文
    2. 运行 convert_to_bio
    3. 将 BIO 标签反向映射回字符范围
    4. 与 gold spans 对比
    """
    # Locate article file
    article_path = os.path.join(article_dir, f'{article_id}.txt')
    if not os.path.exists(article_path):
        print(f'  ❌ 文章不存在: {article_path}')
        return None
    
    # Read as bytes to preserve character offsets
    with open(article_path, 'rb') as f:
        raw = f.read()
    text = raw.decode('utf-8', errors='replace')
    
    gold_spans = gold_spans_dict.get(article_id, [])
    
    # BIO conversion
    encoding, labels = convert_to_bio(text, gold_spans, tokenizer, max_length)
    offset_mapping = encoding['offset_mapping']
    
    # Statistics
    non_o = sum(1 for l in labels if l != 0)
    b_count = sum(1 for l in labels if l % 2 == 1 and l != 0)
    
    if verbose:
        print(f'文章: {article_id}')
        print(f'  文章长度: {len(text)} chars')
        print(f'  Gold spans: {len(gold_spans)}')
        print(f'  Token数 (max_len={max_length}): {sum(1 for s,e in offset_mapping if s != e)} 有效tokens')
        print(f'  BIO非O标签: {non_o} tokens ({b_count} B-tags)')
        
        # Verify: decode BIO labels and check against gold spans
        print(f'  Gold span 验证（前3个）:')
        for span_start, span_end, technique in gold_spans[:3]:
            tech_idx = TECH2IDX.get(technique, -1)
            if tech_idx == -1:
                print(f'    ⚠️  未知技术: {technique}')
                continue
            b_label = 1 + tech_idx * 2
            # 找第一个B标签的位置
            found_b = False
            for tok_idx, (tok_s, tok_e) in enumerate(offset_mapping):
                if labels[tok_idx] == b_label:
                    # 检查是否在span范围内
                    if tok_s >= span_start - 10 and tok_e <= span_end + 10:
                        print(f'    ✅ [{span_start},{span_end}] {technique}: B-tag 在 tok[{tok_idx}] ({tok_s},{tok_e})')
                        found_b = True
                        break
            if not found_b:
                print(f'    ❓ [{span_start},{span_end}] {technique}: 未找到对应B-tag（可能被截断）')
    
    return encoding, labels

# 执行验证（需要先有 gold_en 和文章目录）
en_article_dir = os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'articles')
en_gold_path   = os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt')

if os.path.exists(en_gold_path) and os.path.exists(en_article_dir):
    gold_en = parse_gold_spans(en_gold_path)
    
    # Validate 20 articles
    article_files = [f.replace('.txt', '') for f in os.listdir(en_article_dir) if f.endswith('.txt')]
    verify_ids = [aid for aid in article_files if aid in gold_en][:5]  # Validate first 5 articles
    
    for aid in verify_ids:
        result = verify_bio_conversion(aid, en_article_dir, gold_en, tokenizer_mbert)
        print()
else:
    print('需要先确认数据路径（run cells 9-10 above）')

In [ ]:
# ============================================================
# 14. 搜索 CheckThat 2024 的字符级训练标注
#     这是 Stage 2 token classifier 的训练监督信号
# ============================================================
print('=== 搜索 CheckThat 2024 span 标注文件 ===')

# 已知的 overlap 文件路径
known_overlap = [
    os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt'),
    os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'labels-subtask-3-spans.txt'),
    os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'labels-subtask-3-spans.txt'),
]

for p in known_overlap:
    exists = os.path.exists(p)
    print(f'  {"✅" if exists else "❌"} {p}')
    if exists:
        with open(p, 'rb') as f:
            for i, line in enumerate(f):
                print(f'    Line {i}: {line.decode("utf-8", errors="replace").rstrip()}')
                if i >= 4: break

In [ ]:
# ============================================================
# 15. 统计训练数据量
#     目标：了解每个语言有多少训练样本可用
# ============================================================
def count_training_samples(article_dir, gold_path, max_length=256, tokenizer=None):
    """
    统计某语言的训练样本数量和 span 分布
    """
    if not os.path.exists(article_dir) or not os.path.exists(gold_path):
        print(f'  ❌ 路径不存在')
        return
    
    gold_spans = parse_gold_spans(gold_path)
    articles = [f for f in os.listdir(article_dir) if f.endswith('.txt')]
    
    n_articles = len(articles)
    n_with_spans = sum(1 for a in articles if a.replace('.txt','') in gold_spans)
    all_spans = [s for spans in gold_spans.values() for s in spans]
    
    tech_counts = {}
    for span in all_spans:
        t = span[2]
        tech_counts[t] = tech_counts.get(t, 0) + 1
    
    print(f'  文章总数: {n_articles}')
    print(f'  含span文章: {n_with_spans}')
    print(f'  总span数: {len(all_spans)}')
    print(f'  技术分布 (Top 5): {sorted(tech_counts.items(), key=lambda x: -x[1])[:5]}')

langs = {
    'EN': (os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'articles'),
           os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt')),
    'PL': (os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'articles'),
           os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'labels-subtask-3-spans.txt')),
    'RU': (os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'articles'),
           os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'labels-subtask-3-spans.txt')),
}

for lang, (art_dir, gold_path) in langs.items():
    print(f'\n=== {lang} ===')
    count_training_samples(art_dir, gold_path)

In [ ]:
# ============================================================
# 16. 数据路径汇总与训练集过滤
# [Note] CheckThat! 2024 完整标注在 checkthat24-data-bundle 下，
#         但训练标注中包含测试集文章（overlap_articles），需要剔除。
#         改动：定义 CHECKTHAT24_DIR，并提供过滤函数获取纯训练数据。
# ============================================================

# CheckThat! 2024 完整数据根目录
CHECKTHAT24_DIR = CHECKTHAT24_DIR  # Set in configuration cell above  # CheckThat! 2024 data bundle directory

# 语言目录映射（EN/PL/RU 对应 checkthat24 中的子目录名）
LANG_DIR = {'EN': 'en', 'PL': 'po', 'RU': 'ru'}

# 测试集（SemEval2023 dev ∩ CheckThat2024，字符级金标准）
EVAL_DATA = {
    'EN': {
        'gold_spans': os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt'),
        'articles':   os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'articles'),
    },
    'PL': {
        'gold_spans': os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'labels-subtask-3-spans.txt'),
        'articles':   os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'articles'),
    },
    'RU': {
        'gold_spans': os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'labels-subtask-3-spans.txt'),
        'articles':   os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'articles'),
    },
}

def get_test_article_ids(lang):
    """获取某语言测试集的 article_id 集合（用于从训练集中剔除）"""
    gold_path = EVAL_DATA[lang]['gold_spans']
    ids = set()
    if not os.path.exists(gold_path):
        return ids
    with open(gold_path, 'rb') as f:
        for line in f:
            parts = line.decode('utf-8', errors='replace').rstrip().split('\t')
            if len(parts) >= 1:
                ids.add(parts[0])
    return ids

def load_train_spans(lang, exclude_test=True):
    """
    加载 CheckThat! 2024 字符级训练标注，剔除测试集文章。
    格式：article_id \t technique \t char_start \t char_end
    返回：{article_id: [(start, end, technique), ...]}
    """
    lang_dir = LANG_DIR[lang]
    spans_file = os.path.join(CHECKTHAT24_DIR, lang_dir, 'train-labels-subtask-3-spans.txt')
    articles_dir = os.path.join(CHECKTHAT24_DIR, lang_dir, 'train-articles-subtask-3')

    if not os.path.exists(spans_file):
        print(f'  ❌ 未找到: {spans_file}')
        return {}

    test_ids = get_test_article_ids(lang) if exclude_test else set()
    spans_by_article = {}

    with open(spans_file, 'rb') as f:
        for line in f:
            parts = line.decode('utf-8', errors='replace').rstrip().split('\t')
            if len(parts) != 4:
                continue
            article_id, technique, start, end = parts
            if article_id in test_ids:
                continue  # 剔除测试集文章
            try:
                start, end = int(start), int(end)
            except ValueError:
                continue
            if article_id not in spans_by_article:
                spans_by_article[article_id] = []
            spans_by_article[article_id].append((start, end, technique))

    return spans_by_article

# ── 验证与统计 ────────────────────────────────────────────────
print('=== 训练集统计（剔除测试集文章后）===')
for lang in ['EN', 'PL', 'RU']:
    lang_dir = LANG_DIR[lang]
    spans_file = os.path.join(CHECKTHAT24_DIR, lang_dir, 'train-labels-subtask-3-spans.txt')
    articles_dir = os.path.join(CHECKTHAT24_DIR, lang_dir, 'train-articles-subtask-3')
    test_ids = get_test_article_ids(lang)
    train_spans = load_train_spans(lang, exclude_test=True)
    print(f'  {lang}: spans文件={"✅" if os.path.exists(spans_file) else "❌"}'
          f'  articles目录={"✅" if os.path.exists(articles_dir) else "❌"}')
    print(f'       测试集剔除: {len(test_ids)} 篇文章')
    print(f'       训练集保留: {len(train_spans)} 篇文章, '
          f'{sum(len(v) for v in train_spans.values())} 个 spans')

print()
print('=== 测试集统计 ===')
for lang in ['EN', 'PL', 'RU']:
    test_spans = parse_gold_spans(EVAL_DATA[lang]['gold_spans'])
    n_arts = len(os.listdir(EVAL_DATA[lang]['articles'])) if os.path.exists(EVAL_DATA[lang]['articles']) else 0
    print(f'  {lang}: {n_arts} 篇文章, {len(test_spans)} 篇有标注, '
          f'{sum(len(v) for v in test_spans.values())} 个 spans')

print()
print('Step 1 完成！训练/测试数据路径均已确认，可进入 Step 2 模型框架实现。')
